In [1]:
import sys
sys.path.append('../src')
from data.loader import DataLoader
from data.preprocessor import DataPreprocessor
from data.feature_builder import FeatureBuilder
from embeddings.embedder import FeatureEmbedder
from embeddings.reducer import DimensionalityReducer
from clustering.engine import ClusterEngine
from clustering.optimizer import ClusterOptimizer
from clustering.profiler import ClusterProfiler
from matching.hotel_matcher import HotelMatcher
import plotly.express as px
import pandas as pd

# Data & Embeddings Pipelines
l = DataLoader(data_dir='../data/raw')
c, h = l.get_data()
p = DataPreprocessor()
c_clean, h_clean = p.preprocess_customers(c), p.preprocess_hotels(h)
fb = FeatureBuilder()
vec = fb.build_features(c_clean, h_clean)

embedder = FeatureEmbedder(models_dir='../models')
scaled = embedder.fit_transform(vec)
reducer = DimensionalityReducer(models_dir='../models')
e15, e3 = reducer.fit_transform(scaled)

# Clustering
opt = ClusterOptimizer(models_dir='../models')
res = opt.suggest_optimal_granularity(e15)
print('Granularidad sugerida:', res['optimal_min_cluster_size'])

engine = ClusterEngine(models_dir='../models', min_cluster_size=res['optimal_min_cluster_size'])
labels = engine.fit_predict(e15)

profiler = ClusterProfiler(models_dir='../models')
cards = profiler.generate_profiles(vec, labels, e3)
print(f'Identificados {len(cards)} segmentos útiles.')
for c in cards[:3]:
    print(c)

# Matching Hotel
matcher = HotelMatcher(models_dir='../models')
hotel_serie = h_clean.iloc[0]
coords = matcher.project_hotel_to_embedding_space(hotel_serie)
print(f"\nVector del hotel {hotel_serie['HOTEL_NAME']} proyectado en 3D: {coords}")
affine = matcher.get_affine_clusters_for_hotel(coords, top_n=3)
print(f'Clusters recomendados: {affine}')


/home/mrtn/Desktop/Eurostars/stayprint/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/mrtn/Desktop/Eurostars/stayprint/venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/home/mrtn/Desktop/Eurostars/stayprint/venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Granularidad sugerida: 20
Identificados 5 segmentos útiles.
{'cluster_id': 2, 'name': 'Segmento #2', 'size': 47, 'metrics': {'adr': 240.19, 'score': 7.9, 'age_segment': 3}}
{'cluster_id': 4, 'name': 'Segmento #4', 'size': 30, 'metrics': {'adr': 216.07, 'score': 8.0, 'age_segment': 3}}
{'cluster_id': 3, 'name': 'Segmento #3', 'size': 25, 'metrics': {'adr': 207.32, 'score': 7.8, 'age_segment': 3}}



Vector del hotel Eurostars Torre Sevilla proyectado en 3D: [5.2993717 0.7344406 4.1305027]
Clusters recomendados: [(np.int64(3), np.float64(1.1824312855028334)), (np.int64(2), np.float64(1.388742962159318)), (np.int64(4), np.float64(3.4317457439903087))]
